# California Housing Dataset - Linear Regression in PyTorch

Build a regression model on California Housing from scratch:

StandardScaler → tensors → TensorDataset → DataLoader

3-layer FC network with BatchNorm1d and Dropout

MSELoss + MAE tracking

Train/val/test split

Report final test MAE

In [ ]:
import torch
import torchvision
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
from torch import optim

In [ ]:
# 1. Check for MPS (Apple Silicon GPU)
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS (Apple GPU)")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA")
else:
    device = torch.device("cpu")
    print("Using CPU")

# Load and Scale the Data

In [ ]:
from sklearn.datasets import fetch_california_housing

In [ ]:
housing = fetch_california_housing(data_home="~/Programming/PyTorch")

In [ ]:
print(housing.data.shape, housing.target.shape)

So there are technically 9 total features, one of them being y, which is the price of the house based on the 8 X's. We now know how our neurons are gonna work

In [ ]:
from sklearn.model_selection import train_test_split

X = housing.data
y = housing.target
X_full, X_test, y_full, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_full,y_full,test_size=0.2,random_state=42)

In [ ]:
def pytorch_normalize(X_tensor):
    mean = X_tensor.mean(dim=0, keepdim=True)
    std = X_tensor.std(dim=0, keepdim=True)
    return (X_tensor - mean) / std

In [ ]:
#1, Convert X's to tensors
X_train = torch.tensor(X_train,dtype=torch.float32)
X_val = torch.tensor(X_val,dtype=torch.float32)
X_test = torch.tensor(X_test,dtype=torch.float32)

#2. Convert y's to tensors
y_train = torch.tensor(y_train,dtype=torch.float32).reshape(-1,1)
y_val = torch.tensor(y_val,dtype=torch.float32).reshape(-1,1)
y_test = torch.tensor(y_test,dtype=torch.float32).reshape(-1,1)
# regression has .reshape(-1,1) but classification will have just torch.tensor(source,dtype=torch.long) for the integer labels

#2.5 Normalize
X_train_normalized = pytorch_normalize(X_train)
X_val_normalized = pytorch_normalize(X_val)
X_test_normalized = pytorch_normalize(X_test)

#3. TensorDataset
train_dataset = TensorDataset(X_train_normalized, y_train)
val_dataset = TensorDataset(X_val_normalized, y_val)
test_dataset = TensorDataset(X_test_normalized, y_test)

#4. Dataloaders 
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=64)
test_dataloader = DataLoader(test_dataset, batch_size=64)

# Creating our Regression Model

In [ ]:
class LinearRegression(nn.Module):

    def __init__(self,in_features):

        super().__init__()
        self.in_features = in_features

        #formula: linear --> batchnorm1d --> relu --> dropout
        self.linear_stack = nn.Sequential(
            nn.Linear(self.in_features,32),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Linear(32,64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(64,32),
            nn.BatchNorm1d(32),
            nn.ReLU()
        )

        self.fc = nn.Linear(32,1)

    def forward(self,x):

        x = self.linear_stack(x)
        x = self.fc(x)
        return x

# Optimizer, Loss_Fn, LR Scheduler

In [ ]:
model = LinearRegression(X_train.shape[1])
optimizer = optim.Adam(model.parameters(),lr=0.001)
loss_fn = nn.MSELoss()
lr_scheduler = optim.lr_scheduler.StepLR(optimizer,step_size=20,gamma=0.1) # every 10 epochs, multiply lr by 0.1

# Training our Regression Model

Tracking MAE and MSE

In [ ]:
def train_loop_reg(model, device, optimizer, loss_fn, lr_scheduler, train_dataloader, val_dataloader, train_losses, val_losses, model_save_path, early_stopping, epochs=100, initial_loss=float("inf")):

    best_loss = initial_loss

    for epoch in range(epochs):
        # == TRAINING ==
        model.train()
        train_loss = 0.0
        train_mae = 0.0

        for X_train, y_train in train_dataloader:
            X_train = X_train.to(device)
            y_train = y_train.to(device)

            optimizer.zero_grad()
            y_pred = model(X_train)
            loss = loss_fn(y_pred, y_train)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

            with torch.no_grad():
                mae_train = torch.abs(y_pred-y_train).mean()
                train_mae += mae_train.item()

        avg_train_loss = train_loss / len(train_dataloader)
        avg_train_mae = train_mae / len(train_dataloader)
        train_losses.append(avg_train_loss)
        if ((epoch + 1) == 1) or ((epoch + 1) % 10 == 0):
            print(f"Epoch {epoch+1} ; Train Loss: {avg_train_loss:.4f} ; Train MAE: {avg_train_mae:.4f}")
        
        # == VAL ==
        model.eval()
        val_loss = 0.0
        val_mae = 0.0

        with torch.no_grad():
            for X_val, y_val in val_dataloader:
                X_val = X_val.to(device)
                y_val = y_val.to(device)

                y_pred = model(X_val)
                loss = loss_fn(y_pred, y_val)
                
                val_loss += loss.item()

                mae_val = torch.abs(y_pred-y_val).mean()
                val_mae += mae_val.item()

        avg_val_loss = val_loss / len(val_dataloader)
        avg_val_mae = val_mae / len(val_dataloader)
        val_losses.append(avg_val_loss)
        if ((epoch + 1) == 1) or ((epoch + 1) % 10 == 0):
            print(f"Epoch {epoch+1} ; Val Loss: {avg_val_loss:.4f} ; Val MAE: {avg_val_mae:.4f}")
        
        # == LR Scheduling ==
        lr_scheduler.step()

        # == SAVE BEST MODEL ==
        if avg_val_loss < best_loss:
            best_loss = avg_val_loss
            torch.save(model.state_dict(), model_save_path)

        # == EarlyStopping ==
        if early_stopping is not None:  # just to be safe
            early_stopping(avg_val_loss)
            if early_stopping.early_stop:
                print(f"Early stopping triggered at epoch {epoch+1}")
                break

    return best_loss

In [ ]:
class EarlyStopping:

    def __init__(self,patience=5,min_delta=0):
        self.patience = patience #how many epochs to wait after last improvement 
        self.min_delta = min_delta #minimum change to qualify as improvement
        self.counter = 0 
        self.best_loss = None
        self.early_stop = False

    def __call__(self,val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter+=1
            print(f'EarlyStopping counter: {self.counter}/{self.patience}')
            if self.counter >= self.patience:
                # kill it 
                self.early_stop=True
        else:
            #here is the reset. 
            self.best_loss = val_loss
            self.counter = 0 

In [ ]:
train_losses = []
val_losses = []

In [ ]:
model_save_path_s1 = "best_housing_reg.pth"
model = model.to(device)
loss_fn = loss_fn.to(device)
best_loss_s1 = train_loop_reg(model, device, optimizer, loss_fn, lr_scheduler, train_dataloader, val_dataloader, train_losses, val_losses, model_save_path_s1, EarlyStopping(10,0.01), epochs=100, initial_loss=float("inf"))

In [ ]:
import matplotlib.pyplot as plt

def plot_results(train_losses,val_losses):
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
plot_results(train_losses,val_losses)

# Deploying our Regression Model

In [ ]:
model.load_state_dict(torch.load(model_save_path_s1))

In [ ]:
def test_loop(model,device,loss_fn,test_dataloader):

    model.eval()
    test_loss = 0.0

    with torch.no_grad():
        for X,y in test_dataloader:
            X = X.to(device)
            y = y.to(device)

            y_pred = model(X)
            loss = loss_fn(y_pred,y)

            test_loss += loss.item()

    avg_test_loss = test_loss/len(test_dataloader)

    return avg_test_loss

In [ ]:
avg_test_loss = test_loop(model,device,loss_fn,test_dataloader)
print(avg_test_loss)

If I had the time and mental capacity, here is what I would do to lower the loss:
1. Add more dropout layers 
2. Add maybe one more fully connected layer with more neurons 
3. Add weight decay 
4. lower lr and more often stepping in StepLR